# Word-Dokument-Formatierer mit LangGraph-Agent

**Architektur:** LangGraph-Pipeline → LLM-basierte Stilklassifikation → Deterministische Formatierung

Der Agent liest eine Formatvorlage (House-Style), analysiert Word-Dokumente mit einem lokalen LLM
(via LM Studio) und korrigiert unsaubere Formatierungen intelligent.

### LangGraph-Workflow
```
extract_styles → load_documents → analyze_with_llm → apply_corrections → save_documents
```

### Warum kein Tool-Use?
Das LLM hat hier eine rein **analytische** Rolle: Es liest Absatztext + aktuelle Stil-Info und
entscheidet, welcher Stil semantisch korrekt wäre. Das eigentliche Anwenden übernimmt
deterministischer `python-docx`-Code. Die saubere Trennung **"LLM denkt, Code handelt"** ist robuster.


## Zelle 1: Abhängigkeiten installieren


In [1]:
%pip install python-docx lxml langgraph langchain-core gradio requests --quiet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Zelle 2: Konfiguration


In [ ]:
# ============================================================
# KONFIGURATION – Hier alles Wichtige einstellen
# ============================================================

# --- LM Studio Verbindung ---
API_URL = "http://localhost:1234/v1/chat/completions"
MODELL  = "qwen/qwen3.6-27b"

# --- Pfade (werden über Gradio überschrieben) ---
VORLAGE_PFAD   = "./Formatvorlage_HouseStyle.docx"
EINGABE_ORDNER = "./Inputdocs"
AUSGABE_ORDNER = "./Outputdocs"

# --- Skill-Datei (System-Prompt für das LLM) ---
SKILL_PFAD = "./skills/formatierung_skill.md"

# --- Formatierungs-Optionen ---
CONFIG = {
    "standard_schrift":     "Calibri",
    "standard_groesse_pt":  11.0,
    "tabelle_groesse_pt":   10.0,
    "seitenraender_cm":     {"oben": 1.5, "unten": 1.5, "links": 1.5, "rechts": 1.5},
    "entferne_direkte_absatzformatierung":   True,
    "entferne_direkte_zeichenformatierung":  True,
    "seitenraender_anpassen": True,
    "backup": True,
    # Batch-Größe: So viele Absätze pro LLM-Call
    "batch_size": 40,
    # Max Token für LLM-Antwort
    "max_tokens": 12000,
    # Temperatur (0 = deterministisch)
    "temperature": 0.2,
}

# --- Qwen3: Thinking-Modus deaktivieren ---
# Bei strukturierter JSON-Ausgabe bringt Thinking keinen Mehrwert,
# verlängert aber die Latenz pro Batch erheblich.
# Variante A: In LM Studio unter Model Settings → Chat Template → "enable_thinking": false
# Variante B: Thinking per System-Prompt unterdrücken (wird dem Skill vorangestellt)
THINKING_OFF_PREFIX = "/nothink\n\n"
# Variante C: Falls LM Studio extra_body unterstützt (nicht alle Versionen):
EXTRA_BODY = {}  # z.B. {"chat_template_kwargs": {"enable_thinking": False}}

print("✅ Konfiguration geladen")


## Zelle 3: Imports und Hilfsfunktionen


In [ ]:
import copy
import json
import logging
import re
import shutil
from pathlib import Path
from typing import Any, TypedDict

import requests
from docx import Document
from docx.oxml.ns import qn
from docx.shared import Pt, Cm, RGBColor
from lxml import etree

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
log = logging.getLogger("formatierer")


# ── Skill laden ──────────────────────────────────────────────────────
def lade_skill(pfad: str) -> str:
    """Lädt die Skill-Markdown-Datei als System-Prompt."""
    p = Path(pfad)
    if not p.exists():
        log.warning(f"Skill-Datei nicht gefunden: {pfad} – verwende Fallback-Prompt")
        return (
            "Du bist ein Experte für Word-Formatierung. "
            "Analysiere die Absätze und weise jedem den semantisch korrekten Stil zu. "
            "Antworte NUR mit einem JSON-Array: "
            '[{"index": 0, "style": "Heading 1", "reason": "..."}]'
        )
    return p.read_text(encoding="utf-8")


# ── LLM-Client ───────────────────────────────────────────────────────
def llm_call(system_prompt: str, user_prompt: str, config: dict) -> str:
    """Sendet eine Anfrage an LM Studio und gibt die Antwort zurück."""
    # Thinking-Off-Prefix voranstellen
    full_system = THINKING_OFF_PREFIX + system_prompt

    payload = {
        "model": MODELL,
        "messages": [
            {"role": "system", "content": full_system},
            {"role": "user",   "content": user_prompt},
        ],
        "temperature": config.get("temperature", 0.2),
        "max_tokens":  config.get("max_tokens", 4096),
    }
    # Optionale Extra-Parameter (nur wenn gesetzt)
    if EXTRA_BODY:
        payload.update(EXTRA_BODY)

    try:
        resp = requests.post(API_URL, json=payload, timeout=180)
        resp.raise_for_status()
        data = resp.json()
        # Debug: Antwort-Struktur prüfen
        if "choices" not in data:
            log.error(f"Unerwartete LLM-Antwort (kein 'choices'): {json.dumps(data, ensure_ascii=False)[:300]}")
            return "[]"
        return data["choices"][0]["message"]["content"]
    except requests.exceptions.ConnectionError:
        log.error(f"Verbindung zu LM Studio fehlgeschlagen – läuft der Server auf {API_URL}?")
        return "[]"
    except requests.exceptions.Timeout:
        log.error("LM Studio Timeout (180s) – Modell zu langsam oder hängt?")
        return "[]"
    except Exception as e:
        # Antwort-Body loggen falls verfügbar
        body = ""
        try:
            body = f" | Response: {resp.text[:300]}"
        except:
            pass
        log.error(f"LLM-Fehler: {e}{body}")
        return "[]"


def parse_llm_json(text: str) -> list[dict]:
    """Extrahiert JSON-Array aus der LLM-Antwort (robust gegen Markdown-Fences etc.)."""
    # Sicherheitshalber: Thinking-Block entfernen (falls doch aktiv)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # Markdown-Fences entfernen
    text = re.sub(r"```(?:json)?\s*", "", text)
    text = text.strip()
    # Erstes [ ... letztes ] extrahieren
    start = text.find("[")
    end   = text.rfind("]")
    if start == -1 or end == -1:
        return []
    try:
        return json.loads(text[start:end+1])
    except json.JSONDecodeError as e:
        log.warning(f"JSON-Parse-Fehler: {e}")
        return []


# ── Vorlage-Funktionen ───────────────────────────────────────────────
def lade_vorlage_styles(vorlage_pfad: str) -> tuple[dict, Document, list[str]]:
    """Lädt Styles aus der Vorlage. Gibt (style_map, doc, style_namen) zurück."""
    doc = Document(vorlage_pfad)
    styles = {}
    namen  = []
    for style in doc.styles:
        if style.element is not None:
            key = style.name.lower().replace(" ", "")
            styles[key] = style.element
            namen.append(style.name)
    log.info(f"Vorlage: {len(styles)} Stile geladen")
    return styles, doc, namen


def uebertrage_style_definition(ziel_doc: Document, vorlage_style_element) -> None:
    """Kopiert eine Style-Definition aus der Vorlage ins Ziel-Dokument."""
    styles_element = ziel_doc.part.styles._element
    style_id = vorlage_style_element.get(qn("w:styleId"))
    for existing in styles_element.findall(qn("w:style")):
        if existing.get(qn("w:styleId")) == style_id:
            styles_element.remove(existing)
            break
    styles_element.append(copy.deepcopy(vorlage_style_element))


def normalisiere_run(run, config: dict) -> None:
    """Bereinigt direkte Zeichenformatierung (Fett/Kursiv bleiben erhalten)."""
    if not config.get("entferne_direkte_zeichenformatierung", True):
        return
    run.font.name = config["standard_schrift"]
    run.font.size = Pt(config["standard_groesse_pt"])
    run.font.color.rgb = None
    run.font.highlight_color = None
    rpr = run._r.find(qn("w:rPr"))
    if rpr is not None:
        for tag in [qn("w:rFonts"), qn("w:sz"), qn("w:szCs"),
                    qn("w:color"), qn("w:highlight"), qn("w:spacing"),
                    qn("w:kern"), qn("w:position"), qn("w:vertAlign")]:
            for elem in rpr.findall(tag):
                rpr.remove(elem)


def normalisiere_absatz_format(para) -> None:
    """Entfernt direkte Absatzformatierung (Abstände, Einzüge etc.)."""
    ppr = para._p.find(qn("w:pPr"))
    if ppr is not None:
        for tag in [qn("w:spacing"), qn("w:ind"), qn("w:jc"),
                    qn("w:contextualSpacing"), qn("w:keepLines"),
                    qn("w:keepNext"), qn("w:pageBreakBefore"),
                    qn("w:shd"), qn("w:pBdr")]:
            for elem in ppr.findall(tag):
                ppr.remove(elem)


def normalisiere_seitenraender(doc: Document, config: dict) -> None:
    """Setzt Seitenränder gemäß Konfiguration."""
    r = config["seitenraender_cm"]
    for section in doc.sections:
        section.top_margin    = Cm(r["oben"])
        section.bottom_margin = Cm(r["unten"])
        section.left_margin   = Cm(r["links"])
        section.right_margin  = Cm(r["rechts"])



# ── Absatz-Metadaten für LLM ──────────────────────────────────────
def extrahiere_format_hints(para) -> str:
    """Extrahiert visuelle Formatierungs-Hinweise aus einem Absatz.
    Gibt einen kompakten String zurück, z.B. '[B, 14pt, center]'.
    So sieht das LLM, was der Autor VISUELL beabsichtigt hat —
    unabhängig vom zugewiesenen Stil."""
    hints = []

    # Run-Ebene: Fett, Kursiv, Schriftgröße, Farbe
    # (Analyse des ERSTEN nicht-leeren Runs als Repräsentant)
    for run in para.runs:
        if not run.text.strip():
            continue
        if run.bold:
            hints.append('B')
        if run.italic:
            hints.append('I')
        if run.underline:
            hints.append('U')
        if run.font.size:
            pt = run.font.size.pt
            if pt != 11.0:  # Nur abweichende Größen melden
                hints.append(f'{pt}pt')
        if run.font.color and run.font.color.rgb:
            hints.append(f'color:{run.font.color.rgb}')
        if run.font.name and run.font.name.lower() in ('consolas', 'courier', 'courier new', 'monospace'):
            hints.append('mono')
        break  # Nur ersten relevanten Run analysieren

    # Absatz-Ebene: Ausrichtung, Einzug
    pf = para.paragraph_format
    if pf.alignment is not None:
        from docx.enum.text import WD_ALIGN_PARAGRAPH
        align_map = {
            WD_ALIGN_PARAGRAPH.CENTER: 'center',
            WD_ALIGN_PARAGRAPH.RIGHT: 'right',
            WD_ALIGN_PARAGRAPH.JUSTIFY: 'justify',
        }
        a = align_map.get(pf.alignment)
        if a:
            hints.append(a)
    if pf.left_indent and pf.left_indent > 0:
        hints.append('indent')

    return f'[{", ".join(hints)}]' if hints else '[]'


print("✅ Hilfsfunktionen geladen")


## Zelle 4: LangGraph — State und Nodes

| Node | Aufgabe |
|------|---------|
| `extract_styles` | Formatvorlage parsen, verfügbare Stile extrahieren |
| `load_documents` | Alle .docx aus dem Eingabe-Ordner laden |
| `analyze_with_llm` | Absätze batchweise ans LLM schicken → Stil-Empfehlungen |
| `apply_corrections` | Empfohlene Stile zuweisen + deterministische Bereinigung |
| `save_documents` | Ergebnisse im Ausgabe-Ordner speichern |


In [ ]:
from langgraph.graph import StateGraph, START, END


# ── State-Definition ─────────────────────────────────────────────────
class FormatterState(TypedDict):
    vorlage_pfad:   str
    eingabe_ordner: str
    ausgabe_ordner: str
    config:         dict
    skill_prompt:   str
    # Von Nodes befüllt:
    style_map:      dict           # key → lxml-Element
    style_namen:    list[str]      # Anzeigenamen der verfügbaren Stile
    vorlage_doc:    Any            # Document-Objekt
    dokumente:      list[dict]     # [{pfad, doc, name}, ...]
    korrekturen:    list[dict]     # [{dok_index, aenderungen: [...]}, ...]
    log_messages:   list[str]      # Fortschritts-Log für Gradio


# ── Node 1: extract_styles ──────────────────────────────────────────
def extract_styles(state: FormatterState) -> dict:
    msgs = list(state.get("log_messages", []))
    msgs.append(f"📋 Lade Vorlage: {state['vorlage_pfad']}")

    style_map, vorlage_doc, style_namen = lade_vorlage_styles(state["vorlage_pfad"])

    # Nur Paragraph-Stile für die LLM-Anzeige filtern
    paragraph_stile = [n for n in style_namen
                       if any(n.lower().startswith(p) for p in
                              ["normal", "heading", "title", "subtitle", "list",
                               "caption", "intense", "quote", "book", "footnote",
                               "strong", "toc", "endnote"])]
    msgs.append(f"   → {len(style_map)} Stile total, davon {len(paragraph_stile)} Absatz-Stile")

    return {
        "style_map":    style_map,
        "style_namen":  paragraph_stile,
        "vorlage_doc":  vorlage_doc,
        "log_messages": msgs,
    }


# ── Node 2: load_documents ──────────────────────────────────────────
def load_documents(state: FormatterState) -> dict:
    msgs = list(state.get("log_messages", []))
    ordner = Path(state["eingabe_ordner"])
    msgs.append(f"\n📂 Scanne Ordner: {ordner}")

    dokumente = []
    for pfad in sorted(ordner.glob("*.docx")):
        if pfad.name.startswith("~$"):
            continue
        try:
            doc = Document(pfad)
            dokumente.append({"pfad": str(pfad), "doc": doc, "name": pfad.name})
            msgs.append(f"   ✓ {pfad.name} ({len(doc.paragraphs)} Absätze)")
        except Exception as e:
            msgs.append(f"   ✗ {pfad.name}: {e}")

    if not dokumente:
        msgs.append("⚠️  Keine .docx-Dateien gefunden!")

    return {"dokumente": dokumente, "log_messages": msgs}


# ── Node 3: analyze_with_llm ────────────────────────────────────────
def analyze_with_llm(state: FormatterState) -> dict:
    msgs = list(state.get("log_messages", []))
    config = state["config"]
    skill  = state["skill_prompt"]
    stile  = state["style_namen"]
    batch_size = config.get("batch_size", 40)

    korrekturen = []

    for dok_idx, dok in enumerate(state["dokumente"]):
        doc = dok["doc"]
        msgs.append(f"\n🤖 Analysiere: {dok['name']}")

        alle_aenderungen = []

        # Absätze sammeln MIT Formatierungs-Metadaten
        paragraphs_info = []
        for i, para in enumerate(doc.paragraphs):
            text = para.text.strip()
            stil = para.style.name if para.style else "Normal"
            fmt  = extrahiere_format_hints(para)
            paragraphs_info.append((i, stil, fmt, text))

        # Batches an LLM schicken
        for batch_start in range(0, len(paragraphs_info), batch_size):
            batch = paragraphs_info[batch_start:batch_start + batch_size]

            # User-Prompt bauen (mit Format-Hints)
            zeilen = [f"Verfügbare Stile: {', '.join(stile)}", "", "Absätze:"]
            for idx, stil, fmt, text in batch:
                vorschau = text[:120].replace('"', '\\"') if text else ""
                zeilen.append(f'[{idx}] style={stil} fmt={fmt} | "{vorschau}"')

            user_prompt = "\n".join(zeilen)

            batch_nr = batch_start // batch_size + 1
            total_batches = (len(paragraphs_info) + batch_size - 1) // batch_size
            msgs.append(f"   ⏳ Batch {batch_nr}/{total_batches}: "
                        f"Absätze {batch[0][0]}–{batch[-1][0]} → LLM...")

            # LLM aufrufen
            antwort = llm_call(skill, user_prompt, config)
            aenderungen = parse_llm_json(antwort)

            if aenderungen:
                alle_aenderungen.extend(aenderungen)
                msgs.append(f"   ← {len(aenderungen)} Korrekturen empfangen")
                # Detail-Log: Was wird geändert?
                for ae in aenderungen:
                    idx = ae.get("index", "?")
                    alt = next((s for i, s, _, _ in batch if i == idx), "?")
                    neu = ae.get("style", "?")
                    grund = ae.get("reason", "")
                    if alt != neu:
                        msgs.append(f"      [{idx}] {alt} → {neu}  ({grund})")
            else:
                msgs.append(f"   ← Keine Korrekturen nötig")

        korrekturen.append({
            "dok_index": dok_idx,
            "aenderungen": alle_aenderungen,
        })

    return {"korrekturen": korrekturen, "log_messages": msgs}


# ── Node 4: apply_corrections ───────────────────────────────────────
def apply_corrections(state: FormatterState) -> dict:
    msgs = list(state.get("log_messages", []))
    config    = state["config"]
    style_map = state["style_map"]

    for korr in state["korrekturen"]:
        dok = state["dokumente"][korr["dok_index"]]
        doc = dok["doc"]
        name = dok["name"]
        msgs.append(f"\n✏️  Wende Korrekturen an: {name}")

        # 4a. Style-Definitionen aus Vorlage übertragen
        for style_elem in style_map.values():
            uebertrage_style_definition(doc, style_elem)
        msgs.append("   → Stildefinitionen aus Vorlage übertragen")

        # 4b. LLM-Empfehlungen anwenden
        applied = 0
        skipped = 0
        for ae in korr["aenderungen"]:
            idx   = ae.get("index")
            style = ae.get("style")
            if idx is None or style is None:
                continue
            if idx >= len(doc.paragraphs):
                continue
            para = doc.paragraphs[idx]
            try:
                para.style = doc.styles[style]
                applied += 1
            except KeyError:
                skipped += 1
                msgs.append(f"   ⚠️ Stil '{style}' nicht verfügbar – übersprungen")

        msgs.append(f"   → {applied} Stile korrigiert"
                     + (f", {skipped} übersprungen" if skipped else ""))

        # 4c. Deterministische Bereinigung aller Absätze
        bereinigt = 0
        for para in doc.paragraphs:
            if config.get("entferne_direkte_absatzformatierung", True):
                normalisiere_absatz_format(para)
            for run in para.runs:
                normalisiere_run(run, config)
            bereinigt += 1
        msgs.append(f"   → {bereinigt} Absätze bereinigt (Schrift, Abstände, Farben)")

        # 4d. Tabellen bereinigen
        n_tabellen = len(doc.tables)
        for tabelle in doc.tables:
            for zeile in tabelle.rows:
                for zelle in zeile.cells:
                    for para in zelle.paragraphs:
                        for run in para.runs:
                            normalisiere_run(run, config)
        if n_tabellen:
            msgs.append(f"   → {n_tabellen} Tabelle(n) bereinigt")

        # 4e. Seitenränder
        if config.get("seitenraender_anpassen", True):
            normalisiere_seitenraender(doc, config)
            r = config["seitenraender_cm"]
            msgs.append(f"   → Seitenränder: {r['oben']}/{r['unten']}/{r['links']}/{r['rechts']} cm")

    return {"log_messages": msgs}


# ── Node 5: save_documents ──────────────────────────────────────────
def save_documents(state: FormatterState) -> dict:
    msgs = list(state.get("log_messages", []))
    ausgabe = Path(state["ausgabe_ordner"])
    ausgabe.mkdir(parents=True, exist_ok=True)
    config = state["config"]

    msgs.append(f"\n💾 Speichere nach: {ausgabe}")

    for dok in state["dokumente"]:
        doc  = dok["doc"]
        name = dok["name"]
        ziel = ausgabe / name

        # Backup
        if config.get("backup", True):
            backup_pfad = ausgabe / "backup" / name
            backup_pfad.parent.mkdir(parents=True, exist_ok=True)
            original = Path(dok["pfad"])
            if original.exists():
                shutil.copy2(original, backup_pfad)
                msgs.append(f"   📦 Backup: {backup_pfad}")

        doc.save(ziel)
        msgs.append(f"   ✓ {ziel}")

    msgs.append("\n" + "=" * 50)
    msgs.append("✅ Fertig! Alle Dokumente verarbeitet.")
    return {"log_messages": msgs}


print("✅ LangGraph-Nodes definiert")


## Zelle 5: LangGraph — Graph zusammenbauen


In [ ]:
# ── Graph definieren ──────────────────────────────────────────────────
workflow = StateGraph(FormatterState)

workflow.add_node("extract_styles",    extract_styles)
workflow.add_node("load_documents",    load_documents)
workflow.add_node("analyze_with_llm",  analyze_with_llm)
workflow.add_node("apply_corrections", apply_corrections)
workflow.add_node("save_documents",    save_documents)

workflow.add_edge(START,               "extract_styles")
workflow.add_edge("extract_styles",    "load_documents")
workflow.add_edge("load_documents",    "analyze_with_llm")
workflow.add_edge("analyze_with_llm",  "apply_corrections")
workflow.add_edge("apply_corrections", "save_documents")
workflow.add_edge("save_documents",    END)

graph = workflow.compile()

print("✅ LangGraph-Workflow kompiliert")
print("   START → extract_styles → load_documents → analyze_with_llm")
print("         → apply_corrections → save_documents → END")


## Zelle 6: Streaming-Runner

Nutzt `graph.stream()` statt `graph.invoke()`, damit nach jedem Node
ein Zwischenstand an Gradio ge-yielded werden kann.


In [ ]:
def run_formatter_streaming(vorlage_pfad: str, eingabe_ordner: str,
                            ausgabe_ordner: str, skill_pfad: str = SKILL_PFAD):
    """Generator: Führt den Workflow aus und yielded nach jedem Node das aktuelle Log."""

    skill_prompt = lade_skill(skill_pfad)

    initial_state: FormatterState = {
        "vorlage_pfad":   vorlage_pfad,
        "eingabe_ordner": eingabe_ordner,
        "ausgabe_ordner": ausgabe_ordner,
        "config":         CONFIG,
        "skill_prompt":   skill_prompt,
        "style_map":      {},
        "style_namen":    [],
        "vorlage_doc":    None,
        "dokumente":      [],
        "korrekturen":    [],
        "log_messages":   [],
    }

    node_labels = {
        "extract_styles":    "📋 Vorlage einlesen",
        "load_documents":    "📂 Dokumente laden",
        "analyze_with_llm":  "🤖 LLM-Analyse",
        "apply_corrections": "✏️  Korrekturen anwenden",
        "save_documents":    "💾 Speichern",
    }

    alle_messages = []

    try:
        for event in graph.stream(initial_state, stream_mode="updates"):
            # event ist ein Dict: {node_name: state_update}
            for node_name, update in event.items():
                label = node_labels.get(node_name, node_name)

                # Neue Log-Messages aus diesem Node extrahieren
                neue_msgs = update.get("log_messages", [])

                # Nur die NEUEN Messages (delta)
                delta = neue_msgs[len(alle_messages):]
                alle_messages.extend(delta)

                # Gesamtes Log zusammenbauen und yielden
                yield "\n".join(alle_messages)

    except Exception as e:
        alle_messages.append(f"\n❌ Fehler im Workflow: {e}")
        yield "\n".join(alle_messages)


print("✅ Streaming-Runner bereit")


## Zelle 7: Gradio-Oberfläche

Jeder Node-Abschluss aktualisiert das Log-Fenster live.


In [ ]:
import gradio as gr


def gradio_run(vorlage_datei, eingabe_ordner, ausgabe_ordner, skill_datei):
    """Gradio-Wrapper mit Live-Streaming."""

    # Vorlage-Pfad bestimmen
    if vorlage_datei is not None:
        vorlage_pfad = vorlage_datei if isinstance(vorlage_datei, str) else vorlage_datei.name
    else:
        vorlage_pfad = VORLAGE_PFAD

    # Skill-Pfad bestimmen
    if skill_datei is not None:
        skill_pfad = skill_datei if isinstance(skill_datei, str) else skill_datei.name
    else:
        skill_pfad = SKILL_PFAD

    # Validierung
    if not eingabe_ordner or not eingabe_ordner.strip():
        yield "❌ Bitte einen Eingabe-Ordner angeben."
        return
    if not ausgabe_ordner or not ausgabe_ordner.strip():
        yield "❌ Bitte einen Ausgabe-Ordner angeben."
        return
    if not Path(vorlage_pfad).exists():
        yield f"❌ Vorlage nicht gefunden: {vorlage_pfad}"
        return
    if not Path(eingabe_ordner.strip()).is_dir():
        yield f"❌ Eingabe-Ordner nicht gefunden: {eingabe_ordner}"
        return

    yield "⏳ Workflow gestartet...\n"

    # Streaming-Runner aufrufen – jedes yield aktualisiert das Textfeld
    for log_snapshot in run_formatter_streaming(
        vorlage_pfad, eingabe_ordner.strip(),
        ausgabe_ordner.strip(), skill_pfad
    ):
        yield log_snapshot


# ── Gradio-Interface ─────────────────────────────────────────────────
with gr.Blocks(
    title="Word-Formatierer Agent",
    theme=gr.themes.Soft(),
) as app:
    gr.Markdown("# 📄 Word-Dokument-Formatierer")
    gr.Markdown(
        "Wendet eine Formatvorlage auf Word-Dokumente an. "
        "Ein lokales LLM (via LM Studio) erkennt und korrigiert dabei "
        "unsaubere Stil-Zuweisungen intelligent."
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Eingabe")
            vorlage_input = gr.File(
                label="Formatvorlage (.docx)",
                file_types=[".docx"],
                type="filepath",
            )
            eingabe_input = gr.Textbox(
                label="Eingabe-Ordner (Pfad zu den .docx-Dateien)",
                placeholder="./Inputdocs",
                value=EINGABE_ORDNER,
            )
            ausgabe_input = gr.Textbox(
                label="Ausgabe-Ordner (Ergebnisse werden hier gespeichert)",
                placeholder="./Outputdocs",
                value=AUSGABE_ORDNER,
            )
            skill_input = gr.File(
                label="Skill-Datei (.md) — optional, Standard wird verwendet",
                file_types=[".md"],
                type="filepath",
            )
            run_btn = gr.Button(
                "🚀 Formatierung starten",
                variant="primary",
                size="lg",
            )

        with gr.Column(scale=2):
            gr.Markdown("### 📋 Live-Protokoll")
            output = gr.Textbox(
                label="Verarbeitungs-Log",
                lines=30,
                max_lines=60,
                interactive=False,
            )

    # Click → Streaming-Output
    run_btn.click(
        fn=gradio_run,
        inputs=[vorlage_input, eingabe_input, ausgabe_input, skill_input],
        outputs=output,
    )

    gr.Markdown(
        "---\n"
        f"**LLM:** `{API_URL}` · **Modell:** `{MODELL}` · "
        f"**Skill:** `{SKILL_PFAD}` · **Thinking:** off (/nothink)"
    )


print("✅ Gradio-App definiert")


## Zelle 8: App starten


In [ ]:
app.launch(
    share=False,
    server_name="0.0.0.0",
    server_port=7860,
    inbrowser=True,
)